[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch06-nn-architectures/06-lam-tap-trung-re-hon.ipynb)

# Làm cơ chế tập trung rẻ hơn: chia khối, bộ nhớ đệm KV, chia sẻ khoá và cửa sổ

Chương chỉ ra hai chi phí giới hạn độ dài ngữ cảnh của transformer: ma trận điểm $S \times S$ khi huấn luyện và
xử lý prompt, và bộ nhớ đệm KV phình ra khi sinh token. Nó gọi việc nén bộ nhớ đệm KV và các tối ưu phục vụ liên
quan là những hướng kỹ thuật đang được đẩy mạnh, không phải tinh chỉnh tuỳ chọn. Sổ tay 05 đã mở cơ chế tập trung
(attention mechanism) ra xem bên trong. Sổ tay này đo năm cách làm nó rẻ hơn, và cái giá của từng cách.

**Bạn sẽ làm:**
1. đếm byte mà tập trung đọc và ghi ở bộ nhớ chậm, viết ngây thơ và viết chia khối kiểu FlashAttention;
2. xem `F.scaled_dot_product_attention` trên CPU chọn kernel nào, và nó bỏ được bao nhiêu bộ nhớ;
3. viết vòng sinh token có và không có bộ nhớ đệm KV, rồi đo;
4. tính bộ nhớ đệm KV khi các đầu dùng chung khoá và giá trị (MQA, GQA), rồi huấn luyện để xem mất gì;
5. thử tập trung cửa sổ trượt theo kiểu xen kẽ của GPT-3;
6. thử tập trung tuyến tính: trạng thái cố định như RNN, nhanh hơn ở chuỗi dài, nhưng nhớ kém hơn.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import ProfilerActivity, profile

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]
FP32, FP16 = 4, 2


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f() (giây), sau một lần chạy khởi động."""
    f(); do = []
    for _ in range(lap):
        t0 = time.perf_counter(); f(); do.append(time.perf_counter() - t0)
    return float(np.median(do))


def truc_log(ax):
    """Hai trục log với nhãn số thường (1, 2, 5, 10, ...) thay vì luỹ thừa."""
    from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator
    ax.set_xscale("log")
    ax.set_yscale("log")
    for truc in (ax.xaxis, ax.yaxis):
        truc.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
        truc.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.6g}"))
        truc.set_minor_locator(NullLocator())


print("sẵn sàng")

## 1. Chia khối: cùng phép tính, ít byte hơn

Chương nói rằng với chuỗi dài, chẳng hạn 32k token, riêng ma trận tập trung đã cần hàng gigabyte bộ nhớ cho mỗi tầng,
và chính chi phí đó dẫn tới FlashAttention: chia phép tính thành khối để không bao giờ vật chất hoá cả ma trận ở bộ
nhớ chính (HBM). Sổ tay 03 đã viết vòng lặp chia khối với softmax trực tuyến và kiểm rằng nó cho đúng kết quả. Ở đây
ta đếm thứ nó tiết kiệm: số phần tử phải đọc từ bộ nhớ chậm và ghi trả lại.

Cách viết ngây thơ có ba bước, và mỗi bước ghi kết quả ra bộ nhớ chậm rồi bước sau đọc lại: tính điểm $QK^\top$,
lấy softmax, nhân với $V$. Cách chia khối giữ một khối $B$ hàng của $Q$ trong bộ nhớ nhanh, cho từng khối của $K$ và
$V$ chảy qua, và chỉ ghi đầu ra một lần.

**Dự đoán:** với $S = 1{,}024$, $d = 64$ và khối $B = 128$, cách chia khối bớt được bao nhiêu lần lưu lượng?

In [ ]:
class BoNhoCham:
    """Đếm số phần tử đi qua ranh giới giữa bộ nhớ chậm (HBM) và bộ nhớ nhanh (SRAM)."""
    def __init__(self):
        self.doc, self.ghi = 0, 0

    def doc_vao(self, t):
        self.doc += t.numel()
        return t

    def ghi_ra(self, t):
        self.ghi += t.numel()
        return t


def tap_trung_ngay_tho(q, k, v, mem):
    d = q.shape[1]
    diem = mem.ghi_ra(mem.doc_vao(q) @ mem.doc_vao(k).T / math.sqrt(d))   # S×S điểm ra bộ nhớ chậm
    p = mem.ghi_ra(torch.softmax(mem.doc_vao(diem), -1))                  # đọc lại, softmax, ghi ra
    return mem.ghi_ra(mem.doc_vao(p) @ mem.doc_vao(v))


def tap_trung_chia_khoi(q, k, v, B, mem):
    S, d = q.shape
    o = torch.empty_like(q)
    nhanh = 0                                                     # phần tử lớn nhất cùng lúc ở bộ nhớ nhanh
    for i in range(0, S, B):
        qi = mem.doc_vao(q[i:i + B])
        m = torch.full((qi.shape[0], 1), -math.inf)
        l = torch.zeros(qi.shape[0], 1)
        tich = torch.zeros_like(qi)
        for j in range(0, S, B):
            kj, vj = mem.doc_vao(k[j:j + B]), mem.doc_vao(v[j:j + B])
            s = qi @ kj.T / math.sqrt(d)                          # chỉ một khối B×B, ở bộ nhớ nhanh
            m_moi = torch.maximum(m, s.max(-1, keepdim=True).values)
            p, he_so = torch.exp(s - m_moi), torch.exp(m - m_moi)
            l = he_so * l + p.sum(-1, keepdim=True)
            tich = he_so * tich + p @ vj
            m = m_moi
            nhanh = max(nhanh, qi.numel() + kj.numel() + vj.numel() + tich.numel() + s.numel())
        o[i:i + B] = mem.ghi_ra(tich / l)
    return o, nhanh


def luu_luong_ngay_tho(S, d):
    return 4 * S * d + 4 * S * S


def luu_luong_chia_khoi(S, d, B):
    return 2 * S * d + 2 * S * d * math.ceil(S / B)               # Q và O một lần; K, V một lần cho mỗi khối Q


d_io = 64
g = torch.Generator().manual_seed(SEED)
for S in (256, 512, 1024):
    q, k, v = (torch.randn(S, d_io, generator=g) for _ in range(3))
    m0 = BoNhoCham()
    chuan = tap_trung_ngay_tho(q, k, v, m0)
    assert m0.doc + m0.ghi == luu_luong_ngay_tho(S, d_io)
    for B in (64, 128, 256):
        m1 = BoNhoCham()
        o, nhanh = tap_trung_chia_khoi(q, k, v, B, m1)
        assert torch.allclose(o, chuan, atol=1e-5)
        assert m1.doc + m1.ghi == luu_luong_chia_khoi(S, d_io, B)
        assert nhanh == 4 * B * d_io + B * B
        if S == 1024:
            print(f"S = {S}, B = {B:>3}: ngây thơ {m0.doc + m0.ghi:>10,} phần tử, chia khối {m1.doc + m1.ghi:>9,}"
                  f"  -> ít hơn {(m0.doc + m0.ghi) / (m1.doc + m1.ghi):.2f} lần; bộ nhớ nhanh cần "
                  f"{nhanh * FP16 / 1024:.0f} KB ở FP16")
print("đếm được khớp đúng công thức ở cả 9 cặp (S, B), và kết quả chia khối khớp cách ngây thơ")

Hai công thức vừa được kiểm bằng cách đếm:

$$\text{ngây thơ} = 4Sd + 4S^2, \qquad \text{chia khối} = 2Sd + 2Sd\left\lceil S/B \right\rceil .$$

Cách ngây thơ trả $4S^2$ cho việc ghi rồi đọc lại hai ma trận $S \times S$. Cách chia khối không có số hạng $S^2$
nào chảy qua bộ nhớ chậm: $K$ và $V$ được đọc lại một lần cho mỗi khối $Q$, nên lưu lượng của nó là $S^2 \cdot 2d/B$.
Khi $S$ lớn, tỉ số tiến về $2B/d$: khối càng to càng đỡ, và cái chặn khối là dung lượng của bộ nhớ nhanh, vì nó phải
giữ được $4Bd + B^2$ phần tử cùng lúc. Số phép nhân cộng dồn thì không đổi: chia khối là chính xác, không xấp xỉ.
Đây là phép đếm của riêng sổ tay này, không phải của sách.

Ô dưới đưa hai công thức tới 32k token, như ví dụ của chương.

In [ ]:
S_DAI = sach(32, trich="for example, 32k tokens") * 1024           # "32k" ở đây lấy là 32 × 1,024
print(f"{S_DAI:,} token: một ma trận điểm FP16 cho một đầu là {S_DAI * S_DAI * FP16 / 1e9:.2f} GB, mỗi tầng")
theo_sach("bộ nhớ điểm khi chuỗi dài từ 512 lên 2,048 (lần)", (2048 / 512) ** 2, sach=16,
          nguon="2.048 so với 512 = 16×")

cac_S = [2 ** i for i in range(8, 16)]
fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.plot(cac_S, [luu_luong_ngay_tho(S, d_io) * FP16 / 1e6 for S in cac_S], "-", color=MAU[1], label="ngây thơ")
for B, mau in zip((64, 128, 256), (MAU[0], MAU[2], MAU[4])):
    ax.plot(cac_S, [luu_luong_chia_khoi(S, d_io, B) * FP16 / 1e6 for S in cac_S], "-", color=mau,
            label=f"chia khối, B = {B}")
da_dem = [luu_luong_ngay_tho(S, d_io) for S in (256, 512, 1024)]
da_dem += [luu_luong_chia_khoi(S, d_io, B) for B in (64, 128, 256) for S in (256, 512, 1024)]
ax.plot([256, 512, 1024] * 4, [n * FP16 / 1e6 for n in da_dem], "o", color="k", mfc="none",
        label="đã đếm bằng BoNhoCham")
truc_log(ax)
ax.set_xlabel("độ dài chuỗi S (token)")
ax.set_ylabel("lưu lượng bộ nhớ chậm (MB, FP16)")
ax.set_title("Một đầu tập trung, d = 64: byte qua bộ nhớ chậm (đếm, không đo)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Cả hai đường đều đi lên theo bậc hai, vì $K$ và $V$ vẫn được đọc lại $S/B$ lần. Khác ở hằng số: khoảng cách giữa
chúng là hệ số $2B/d$, và nó không đổi khi chuỗi dài ra. Thứ chia khối thực sự xoá bỏ là bộ nhớ: không có ma trận
$S \times S$ nào phải tồn tại, ở bất kỳ độ dài nào.

## 2. `F.scaled_dot_product_attention` làm việc này cho bạn

PyTorch đặt nhiều kernel tập trung sau một hàm duy nhất, và chọn kernel theo phần cứng và đối số. Ô dưới hỏi trình
phân tích (profiler) xem trên CPU nó chạy kernel nào, rồi so lần cấp phát lớn nhất của nó với cách viết bằng tay ở
$S = 2{,}048$ và 4 đầu.

**Dự đoán:** cách viết bằng tay cấp phát một khối lớn cỡ nào? Hàm có sẵn thì sao?

In [ ]:
def cap_phat(f):
    """Lần cấp phát lớn nhất của một phép toán (kể cả các phép con), và tên các kernel tập trung đã chạy."""
    with profile(activities=[ProfilerActivity.CPU], profile_memory=True, acc_events=True) as p:
        f()
    su_kien = p.events()
    return max(e.cpu_memory_usage for e in su_kien), sorted({e.name for e in su_kien if "attention" in e.name})


S_sd, H_sd = 2048, 4
g = torch.Generator().manual_seed(SEED)
q, k, v = (torch.randn(1, H_sd, S_sd, d_io, generator=g) for _ in range(3))
bang_tay = lambda: torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(d_io), -1) @ v
co_san = lambda: F.scaled_dot_product_attention(q, k, v)
print(f"lệch lớn nhất giữa hai cách: {(bang_tay() - co_san()).abs().max():.1e}")
for ten, f in (("bằng tay", bang_tay), ("scaled_dot_product_attention", co_san)):
    lon, kernel = cap_phat(f)
    print(f"{ten:>29}: cấp phát lớn nhất {lon / 1e6:6.1f} MB  kernel: {', '.join(kernel) or '(không có, chỉ matmul và softmax)'}")
print(f"{'':>29}  để so: {H_sd} ma trận điểm FP32 = {H_sd * S_sd * S_sd * FP32 / 1e6:.1f} MB, "
      f"đầu ra = {H_sd * S_sd * d_io * FP32 / 1e6:.1f} MB")

print(f"\n{'S':>6} {'bằng tay (ms)':>14} {'có sẵn (ms)':>12}")
for S in (512, 1024, 2048, 4096):
    q, k, v = (torch.randn(1, H_sd, S, d_io, generator=g) for _ in range(3))
    t_tay = thoi_gian(lambda: torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(d_io), -1) @ v, lap=3)
    t_co = thoi_gian(lambda: F.scaled_dot_product_attention(q, k, v), lap=3)
    print(f"{S:>6} {1e3 * t_tay:>14.1f} {1e3 * t_co:>12.1f}")
do_tren_may("số luồng CPU khi đo", torch.get_num_threads())

Trên CPU, PyTorch cũng có một kernel chia khối (`_scaled_dot_product_flash_attention_for_cpu`). Cách viết bằng tay
cấp phát nguyên ma trận điểm, còn kernel chia khối chỉ cấp phát cỡ đầu ra. Thời gian thì chỉ chênh nhau chút ít:
trên CPU, phép nhân ma trận và hàm mũ chiếm gần hết thời gian, còn việc đọc ghi ma trận điểm chỉ là phần nhỏ, nên
cái lợi chính ở đây là bộ nhớ. Trên GPU thì ngược lại: phép tính nhanh hơn nhiều so với đọc ghi HBM, nên lưu lượng
ở phần 1 mới là thứ quyết định thời gian.

## 3. Bộ nhớ đệm KV: đừng tính lại thứ đã tính

Khi sinh token, mỗi bước thêm đúng một token. Không có bộ nhớ đệm, bước thứ $t$ chạy lại cả mô hình trên $t$ token
để lấy đầu ra ở token cuối. Nhưng khoá và giá trị của $t - 1$ token đầu không đổi giữa các bước, vì mặt nạ nhân quả
không cho chúng nhìn token mới. Bộ nhớ đệm KV giữ chúng lại, nên mỗi bước chỉ chiếu token mới, rồi để truy vấn của
nó nhìn vào mọi khoá đã có.

Ô dưới dựng mô hình dùng cho cả sổ tay. Mỗi tầng tập trung có: số đầu truy vấn, số đầu khoá và giá trị
(`kv`, phần 4), một cửa sổ (`cua_so`, phần 5, 0 là nhìn hết), kiểu tập trung (softmax hay tuyến tính, phần 6), và
một bộ nhớ đệm KV tuỳ chọn.

In [ ]:
def hinh_sin(T, d):
    """Mã hoá vị trí hình sin, cộng vào embedding trước tầng đầu tiên."""
    vi = torch.arange(T, dtype=torch.float32)[:, None]
    tan_so = 10000 ** (-torch.arange(0, d, 2, dtype=torch.float32) / d)
    pe = torch.zeros(T, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(vi * tan_so), torch.cos(vi * tan_so)
    return pe


def mat_na(bat_dau, T, T_khoa, cua_so):
    """True là được nhìn. Truy vấn ở các vị trí bat_dau..bat_dau+T−1, khoá ở 0..T_khoa−1:
    nhân quả, và chỉ `cua_so` khoá gần nhất nếu cua_so > 0."""
    i = torch.arange(bat_dau, bat_dau + T)[:, None]
    j = torch.arange(T_khoa)[None]
    duoc = j <= i
    if cua_so:
        duoc &= i - j < cua_so
    return duoc


class TapTrung(nn.Module):
    def __init__(self, d, dau, kv, cua_so, kieu):
        super().__init__()
        self.dau, self.kv, self.d_dau, self.cua_so, self.kieu = dau, kv, d // dau, cua_so, kieu
        self.q = nn.Linear(d, d)
        self.kv_chieu = nn.Linear(d, 2 * kv * self.d_dau)     # kv đầu khoá và kv đầu giá trị
        self.ra = nn.Linear(d, d)

    def forward(self, x, bo_dem=None, bat_dau=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.dau, self.d_dau).transpose(1, 2)
        k, v = self.kv_chieu(x).view(B, T, 2, self.kv, self.d_dau).permute(2, 0, 3, 1, 4)
        if bo_dem is not None:                                 # nối khoá, giá trị mới vào những cái đã giữ
            if "k" in bo_dem:
                k, v = torch.cat([bo_dem["k"], k], 2), torch.cat([bo_dem["v"], v], 2)
            bo_dem["k"], bo_dem["v"] = k, v
        if self.kieu == "tuyen_tinh":                          # phần 6; viết ở dạng bậc hai cho ngắn
            fq, fk = F.elu(q) + 1, F.elu(k) + 1
            w = (fq @ fk.transpose(-1, -2)).tril()
            a = (w @ v) / w.sum(-1, keepdim=True)
        else:
            a = F.scaled_dot_product_attention(q, k, v, attn_mask=mat_na(bat_dau, T, k.shape[2], self.cua_so),
                                               enable_gqa=self.kv != self.dau)
        return self.ra(a.transpose(1, 2).reshape(B, T, -1))


class Khoi(nn.Module):
    def __init__(self, d, dau, kv, cua_so, d_ff, kieu):
        super().__init__()
        self.chuan1, self.chuan2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.tap_trung = TapTrung(d, dau, kv, cua_so, kieu)
        self.truyen_thang = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))

    def forward(self, x, bo_dem=None, bat_dau=0):
        x = x + self.tap_trung(self.chuan1(x), bo_dem, bat_dau)
        return x + self.truyen_thang(self.chuan2(x))


class MoHinh(nn.Module):
    """`cua_so` có một phần tử cho mỗi tầng; độ dài của nó là số tầng."""
    def __init__(self, V, d=64, dau=4, kv=None, cua_so=(0, 0), d_ff=128, kieu="softmax"):
        super().__init__()
        self.V, self.d = V, d
        self.embedding = nn.Embedding(V, d)
        self.khoi = nn.ModuleList(Khoi(d, dau, kv or dau, w, d_ff, kieu) for w in cua_so)
        self.chuan = nn.LayerNorm(d)
        self.ra = nn.Linear(d, V)

    def forward(self, x, bo_dem=None, bat_dau=0):
        h = self.embedding(x) + hinh_sin(bat_dau + x.shape[1], self.d)[bat_dau:]
        for i, khoi in enumerate(self.khoi):
            h = khoi(h, None if bo_dem is None else bo_dem[i], bat_dau)
        return self.ra(self.chuan(h))


print("sẵn sàng")

Một mô hình 4 tầng, $d = 256$, trọng số ngẫu nhiên (để đo tốc độ thì không cần huấn luyện). Nó sinh 240 token sau
một prompt 16 token, theo hai cách: tính lại cả chuỗi mỗi bước, và giữ bộ nhớ đệm KV. Cả hai lấy token có xác suất
cao nhất, nên phải ra đúng một chuỗi.

**Dự đoán:** thời gian sinh một token thay đổi thế nào theo vị trí, ở mỗi cách?

In [ ]:
N_SINH, N_PROMPT = 240, 16


def sinh_tinh_lai(m, prompt):
    chuoi, moc = prompt, []
    for _ in range(N_SINH):
        t0 = time.perf_counter()
        tiep = m(chuoi)[:, -1].argmax(-1, keepdim=True)        # chạy lại cả chuỗi
        moc.append(time.perf_counter() - t0)
        chuoi = torch.cat([chuoi, tiep], 1)
    return chuoi, moc


def sinh_co_bo_dem(m, prompt):
    bo_dem = [{} for _ in m.khoi]
    t0 = time.perf_counter()
    tiep = m(prompt, bo_dem)[:, -1].argmax(-1, keepdim=True)  # xử lý prompt một lần
    moc = [time.perf_counter() - t0]
    chuoi = torch.cat([prompt, tiep], 1)
    for _ in range(N_SINH - 1):
        t0 = time.perf_counter()
        tiep = m(chuoi[:, -1:], bo_dem, bat_dau=chuoi.shape[1] - 1)[:, -1].argmax(-1, keepdim=True)
        moc.append(time.perf_counter() - t0)
        chuoi = torch.cat([chuoi, tiep], 1)
    return chuoi, moc, bo_dem


dat_hat_giong()
m_sinh = MoHinh(64, d=256, dau=4, cua_so=(0,) * 4, d_ff=1024).eval()
prompt = torch.randint(0, 64, (1, N_PROMPT), generator=torch.Generator().manual_seed(1))
with torch.inference_mode():
    chuoi_a, moc_a = sinh_tinh_lai(m_sinh, prompt)
    chuoi_b, moc_b, bo_dem = sinh_co_bo_dem(m_sinh, prompt)
assert torch.equal(chuoi_a, chuoi_b)
print(f"hai cách sinh ra đúng một chuỗi {chuoi_a.shape[1]} token")
do_tren_may("tính lại mỗi bước: tổng thời gian", round(sum(moc_a), 2), "s")
do_tren_may("có bộ nhớ đệm KV: tổng thời gian", round(sum(moc_b), 2), "s")
byte_do = sum(c["k"].numel() + c["v"].numel() for c in bo_dem) * FP32
T_cuoi = bo_dem[0]["k"].shape[2]
cong_thuc = 4 * 2 * 4 * T_cuoi * (256 // 4) * FP32                 # N_L · 2 · N_heads · S · d_head · byte
print(f"bộ nhớ đệm sau {T_cuoi} token: {byte_do:,} byte; công thức N_L · 2 · N_heads · S · d_head · 4 byte = {cong_thuc:,}")
assert byte_do == cong_thuc

fig, ax = plt.subplots(figsize=(6.8, 3.6))
vi_tri = np.arange(N_PROMPT, N_PROMPT + N_SINH).reshape(-1, 16).mean(1)
for moc, ten, mau in ((moc_a, "tính lại cả chuỗi mỗi bước", MAU[1]), (moc_b, "có bộ nhớ đệm KV", MAU[0])):
    nhom = np.array(moc).reshape(-1, 16)                        # trung vị theo nhóm 16 token cho đỡ nhiễu
    ax.plot(vi_tri, 1e3 * np.median(nhom, 1), "o-", color=mau, label=ten)
ax.set_ylim(bottom=0)
ax.set_xlabel("vị trí của token được sinh")
ax.set_ylabel("thời gian sinh một token (ms)")
ax.set_title("Sinh token, 4 tầng, d = 256 (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Không có bộ nhớ đệm, mỗi token mới đắt hơn token trước, vì phải chạy lại cả chuỗi; tổng chi phí sinh $n$ token vì
thế tăng theo bậc hai. Có bộ nhớ đệm, chi phí một token gần như phẳng: phép chiếu và tầng truyền thẳng chỉ chạy
trên một token, còn phần tăng dần (truy vấn mới nhìn mọi khoá đã giữ) còn nhỏ ở độ dài này. Đổi lại, bộ nhớ đệm
chiếm chỗ, đúng theo công thức của chương: $N_L \times 2 \times N_{\text{heads}} \times S \times d_{\text{head}}$
phần tử cho mỗi yêu cầu, với hệ số 2 là khoá và giá trị.

## 4. Chia sẻ khoá và giá trị giữa các đầu: MQA và GQA

Mỗi đầu truy vấn cần khoá và giá trị của riêng nó, nên bộ nhớ đệm tỉ lệ với số đầu. Hai biến thể không có trong
chương cắt đúng thừa số đó:

* **Tập trung nhiều truy vấn** (multi-query attention, MQA; Shazeer, 2019): mọi đầu truy vấn dùng chung **một** đầu
  khoá và một đầu giá trị.
* **Tập trung theo nhóm truy vấn** (grouped-query attention, GQA; Ainslie và cộng sự, 2023): các đầu truy vấn chia
  thành $G$ nhóm, mỗi nhóm dùng chung một đầu khoá và một đầu giá trị. $G$ bằng số đầu là tập trung nhiều đầu thường,
  $G = 1$ là MQA.

Ô dưới đưa công thức của chương vào ví dụ 32 tầng, 32 đầu của nó, rồi thay số đầu khoá và giá trị bằng $G$.
Đây là phép tính của riêng sổ tay này, không phải của sách.

**Dự đoán:** với GQA 8 nhóm, bộ nhớ đệm của một yêu cầu 2,048 token còn bao nhiêu?

In [ ]:
N_L = sach(32, nguon="32 tầng · 32 đầu")
N_DAU = sach(32, nguon="32 tầng · 32 đầu")
D_DAU = sach(128, trich="128-dimensional heads")
S_YC = sach(2048, trich="2,048-token sequences")
HE_SO = sach(2, trich="the factor 2 stores keys and values")


def bo_dem_kv(S, G):
    """Byte FP16 của bộ nhớ đệm KV cho một yêu cầu, với G đầu khoá/giá trị."""
    return N_L * HE_SO * G * S * D_DAU * FP16


theo_sach("bộ nhớ đệm KV của một yêu cầu, 32 đầu khoá/giá trị (GB)", bo_dem_kv(S_YC, N_DAU) / 1e9, sach=1,
          trich="1 GB of KV cache", tol=0.1)
S_200K = sach(200_000, nguon="512–2K · 128K · 200K · 1 triệu+")
MOC_100 = sach(100, trich="this cache can become massive (for example, 100+ GB)")
THIET_BI = sach(80, trich="Those values do not exhaust a 80 GB device")
print(f"\n{'G':>4} {'tên':>22} {'mỗi token (KB)':>15} {'2,048 token (GB)':>17} {'200K token (GB)':>16}")
for G, ten in ((32, "nhiều đầu (MHA)"), (8, "GQA, 8 nhóm"), (4, "GQA, 4 nhóm"), (1, "MQA")):
    print(f"{G:>4} {ten:>22} {bo_dem_kv(1, G) / 1024:>15,.0f} {bo_dem_kv(S_YC, G) / 1e9:>17.3f} "
          f"{bo_dem_kv(S_200K, G) / 1e9:>16.1f}")
print(f"\nchương: bộ nhớ đệm có thể lên tới hơn {MOC_100} GB ở ngữ cảnh dài; ở {S_200K:,} token, 32 đầu cần "
      f"{bo_dem_kv(S_200K, N_DAU) / 1e9:.1f} GB -> {'khớp' if bo_dem_kv(S_200K, N_DAU) / 1e9 > MOC_100 else 'không khớp'}")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
cac_ngu_canh = np.array([2 ** i for i in range(9, 21)])
for (G, ten), mau in zip(((32, "nhiều đầu, G = 32"), (8, "GQA, G = 8"), (1, "MQA, G = 1")), MAU):
    ax.plot(cac_ngu_canh, [bo_dem_kv(S, G) / 1e9 for S in cac_ngu_canh], "-", color=mau, label=ten)
ax.axhline(THIET_BI, color="0.5", lw=0.8, ls="--", label=f"một thiết bị {THIET_BI} GB")
truc_log(ax)
ax.set_xlabel("độ dài ngữ cảnh (token)")
ax.set_ylabel("bộ nhớ đệm KV một yêu cầu (GB, FP16)")
ax.set_xticks([2 ** i for i in range(9, 21, 2)], ["512", "2K", "8K", "32K", "128K", "512K"])
ax.set_title("Ví dụ 32 tầng, 32 đầu, đầu 128 chiều của chương")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Bộ nhớ đệm giảm đúng $N_{\text{heads}}/G$ lần, còn số phép tính của điểm thì không: mỗi đầu truy vấn vẫn so với mọi
khoá. `F.scaled_dot_product_attention` nhận thẳng $G$ đầu khoá với `enable_gqa=True`. Ô dưới kiểm rằng nó đúng
bằng việc lặp mỗi đầu khoá cho cả nhóm của nó rồi chạy tập trung nhiều đầu thường.

In [ ]:
g = torch.Generator().manual_seed(SEED)
q8 = torch.randn(1, 8, 64, 16, generator=g)                    # 8 đầu truy vấn
k2, v2 = torch.randn(1, 2, 64, 16, generator=g), torch.randn(1, 2, 64, 16, generator=g)   # 2 đầu khoá/giá trị
lap_lai = F.scaled_dot_product_attention(q8, k2.repeat_interleave(4, 1), v2.repeat_interleave(4, 1), is_causal=True)
gqa = F.scaled_dot_product_attention(q8, k2, v2, is_causal=True, enable_gqa=True)
print(f"enable_gqa so với lặp đầu khoá: lệch lớn nhất {(lap_lai - gqa).abs().max():.1e}")
for kv in (4, 2, 1):
    dat_hat_giong()
    m = MoHinh(64, d=256, dau=4, kv=kv, cua_so=(0,) * 4, d_ff=1024).eval()
    with torch.inference_mode():
        _, moc, bd = sinh_co_bo_dem(m, prompt)
    print(f"mô hình sinh token ở phần 3 với {kv} đầu khoá/giá trị: bộ nhớ đệm "
          f"{sum(c['k'].numel() + c['v'].numel() for c in bd) * FP32 / 1e6:.2f} MB")

Bộ nhớ thì rẻ đi, nhưng mô hình mất gì? Để thấy được, cần một bài toán mà tập trung thực sự phải tra cứu. Bài
**nhớ lại** (recall): ngữ cảnh là 8 cặp (khoá, giá trị) ngẫu nhiên, khoá lấy từ 16 token, giá trị từ 16 token khác;
sau đó là 4 câu hỏi, mỗi câu là một khoá đã xuất hiện, và mô hình phải trả ra giá trị đi kèm nó. Chỉ chấm ở 4 vị trí
câu hỏi. Đoán bừa một giá trị có trong ngữ cảnh thì đúng 1 phần 8.

Ba cách có cùng mô hình 2 tầng, 4 đầu truy vấn, $d = 64$:

* tập trung nhiều đầu thường, 4 đầu khoá và giá trị, huấn luyện từ đầu;
* MQA, 1 đầu khoá và giá trị, huấn luyện từ đầu;
* **huấn luyện tiếp** (uptraining, cách bài báo GQA đề xuất): lấy mô hình nhiều đầu đã học xong, gộp các đầu khoá
  và giá trị trong mỗi nhóm bằng cách lấy trung bình trọng số chiếu của chúng, rồi huấn luyện thêm một ít.

**Dự đoán:** ngay sau khi gộp, trước khi huấn luyện thêm, mô hình MQA còn trả lời đúng bao nhiêu?

In [ ]:
NK, P, NQ = 16, 8, 4                    # 16 khoá có thể, 8 cặp trong ngữ cảnh, 4 câu hỏi
V_NL = 2 * NK                           # token 0–15 là khoá, 16–31 là giá trị
VI_TRI_HOI = torch.arange(2 * P, 2 * P + 2 * NQ, 2)     # vị trí của 4 khoá được hỏi


def bai_nho_lai(B, g):
    khoa = torch.argsort(torch.rand(B, NK, generator=g), 1)[:, :P]
    gia_tri = torch.randint(NK, 2 * NK, (B, P), generator=g)
    ngu_canh = torch.stack([khoa, gia_tri], 2).reshape(B, -1)
    hoi = torch.randint(0, P, (B, NQ), generator=g)
    cau_hoi = torch.stack([khoa.gather(1, hoi), gia_tri.gather(1, hoi)], 2).reshape(B, -1)
    chuoi = torch.cat([ngu_canh, cau_hoi], 1)
    y = torch.full((B, chuoi.shape[1] - 1), -100)
    y[:, VI_TRI_HOI] = chuoi[:, VI_TRI_HOI + 1]          # chỉ chấm giá trị đi sau mỗi khoá được hỏi
    return chuoi[:, :-1], y


def do_chinh_xac(mo_hinh, x, y):
    with torch.no_grad():
        du_doan = mo_hinh(x).argmax(-1)
    chon = y != -100
    return (du_doan[chon] == y[chon]).float().mean().item()


def huan_luyen(mo_hinh, buoc, moi=100, lr=3e-3, hat=0):
    """Adam, entropy chéo ở các vị trí câu hỏi. Trả về độ chính xác kiểm tra sau mỗi `moi` bước, và số giây."""
    opt = torch.optim.Adam(mo_hinh.parameters(), lr=lr, foreach=False)
    g = torch.Generator().manual_seed(hat)
    duong = []
    t0 = time.perf_counter()
    for b in range(1, buoc + 1):
        x, y = bai_nho_lai(64, g)
        loss = F.cross_entropy(mo_hinh(x).reshape(-1, V_NL), y.reshape(-1), ignore_index=-100)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if b % moi == 0:
            duong.append(do_chinh_xac(mo_hinh, *KIEM_TRA))
    return duong, time.perf_counter() - t0


KIEM_TRA = bai_nho_lai(1000, torch.Generator().manual_seed(1))
duong_nl = {}
dat_hat_giong()
mha = MoHinh(V_NL)
duong_nl["nhiều đầu"], giay = huan_luyen(mha, 1000)
do_tren_may("nhiều đầu, từ đầu: độ chính xác sau 1,000 bước", round(duong_nl["nhiều đầu"][-1], 3))
do_tren_may("nhiều đầu, từ đầu: thời gian", round(giay, 1), "s")
dat_hat_giong()
duong_nl["MQA"], giay = huan_luyen(MoHinh(V_NL, kv=1), 1000)
do_tren_may("MQA, từ đầu: độ chính xác sau 1,000 bước", round(duong_nl["MQA"][-1], 3))
do_tren_may("MQA, từ đầu: thời gian", round(giay, 1), "s")


def gop_dau(nguon, G):
    """Mô hình G đầu khoá/giá trị, lấy mọi trọng số từ `nguon`; phép chiếu khoá và giá trị của mỗi nhóm đầu là
    trung bình của các đầu trong nhóm."""
    m = MoHinh(V_NL, kv=G)
    m.load_state_dict({k: v for k, v in nguon.state_dict().items() if "kv_chieu" not in k}, strict=False)
    for dich, goc in zip(m.khoi, nguon.khoi):
        H, dh, d = goc.tap_trung.dau, goc.tap_trung.d_dau, nguon.d
        W = goc.tap_trung.kv_chieu.weight.data.view(2, G, H // G, dh, d)
        b = goc.tap_trung.kv_chieu.bias.data.view(2, G, H // G, dh)
        dich.tap_trung.kv_chieu.weight.data = W.mean(2).reshape(-1, d).clone()
        dich.tap_trung.kv_chieu.bias.data = b.mean(2).reshape(-1).clone()
    return m


tiep_tuc = {}
for G in (2, 1):
    m = gop_dau(mha, G)
    truoc = do_chinh_xac(m, *KIEM_TRA)
    duong, giay = huan_luyen(m, 300, moi=50, lr=1e-3, hat=7)
    tiep_tuc[G] = [truoc] + duong
    ten = "MQA" if G == 1 else f"GQA, {G} nhóm"
    do_tren_may(f"{ten}, gộp từ mô hình nhiều đầu: độ chính xác ngay sau khi gộp", round(truoc, 3))
    do_tren_may(f"{ten}, gộp từ mô hình nhiều đầu: sau 300 bước huấn luyện tiếp", round(duong[-1], 3))
    do_tren_may(f"{ten}, huấn luyện tiếp: thời gian", round(giay, 1), "s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.8, 3.6), sharey=True)
buoc = np.arange(100, 1001, 100)
ax1.plot(buoc, duong_nl["nhiều đầu"], "o-", color=MAU[0], label="nhiều đầu: 4 đầu khoá/giá trị")
ax1.plot(buoc, duong_nl["MQA"], "o-", color=MAU[1], label="MQA: 1 đầu khoá/giá trị")
ax1.set_xlabel("bước huấn luyện")
ax1.set_ylabel("độ chính xác ở câu hỏi")
ax1.set_title("Huấn luyện từ đầu", fontsize=10)
for G, mau in ((2, MAU[2]), (1, MAU[1])):
    ax2.plot(np.arange(0, 301, 50), tiep_tuc[G], "o-", color=mau, label="MQA" if G == 1 else f"GQA, {G} nhóm")
ax2.set_xlabel("bước huấn luyện tiếp, sau khi gộp")
ax2.set_title("Gộp đầu từ mô hình nhiều đầu đã học", fontsize=10)
for ax in (ax1, ax2):
    ax.axhline(1 / P, color="0.5", lw=0.8, ls="--", label="đoán bừa trong ngữ cảnh: 1/8")
    ax.set_ylim(0, 1.05)
ax1.legend(loc="center left", fontsize=8)
ax2.legend(loc="center right", fontsize=8)
fig.suptitle("Bài nhớ lại, 8 cặp khoá-giá trị (đo trên máy này)", fontsize=10)
plt.tight_layout()
plt.show()

Ở bài nhỏ này, MQA học từ đầu vẫn tới đích, chỉ chậm hơn: một đầu khoá và giá trị là đủ chỗ cho 8 cặp. Gộp đầu thì
làm hỏng mô hình ngay lập tức, nhiều hơn khi gộp mạnh hơn, vì mỗi đầu đã học một phép chiếu riêng và trung bình của
chúng không còn là phép chiếu nào cả. Nhưng một ít huấn luyện tiếp là kéo nó về, rẻ hơn hẳn huấn luyện lại. Ở mô
hình lớn, cái giá chất lượng của MQA thường lộ rõ hơn, và GQA là điểm giữa mà nhiều mô hình mở chọn.

## 5. Cửa sổ trượt: mỗi token chỉ nhìn gần

Chương kể rằng GPT-3 xen kẽ tập trung dày đặc với tập trung thưa theo dải cục bộ. Tập trung theo dải, hay tập trung
cửa sổ trượt (sliding-window attention), cho mỗi token chỉ nhìn $w$ token gần nhất trước nó. Số điểm phải tính giảm
từ cỡ $S^2/2$ xuống $S \cdot w$, và bộ nhớ đệm KV của tầng đó không bao giờ vượt $w$ token.

Cái giá: thông tin chỉ đi được $w - 1$ vị trí mỗi tầng. Ô dưới huấn luyện bài nhớ lại với cửa sổ 4 token ở **cả
hai** tầng, và với cửa sổ 4 ở tầng đầu rồi một tầng dày đặc, rồi so với mô hình dày đặc ở phần 4.

**Dự đoán:** ở bài này, câu hỏi cuối cách cặp khoá-giá trị đầu tiên 22 token. Cách nào trả lời được?

In [ ]:
cua_so_vd, S_vd = 4096, sach(32, trich="for example, 32k tokens") * 1024
print(f"điểm phải tính ở {S_vd:,} token: dày đặc nhân quả {S_vd * (S_vd + 1) // 2:,}, cửa sổ {cua_so_vd:,} "
      f"khoảng {S_vd * cua_so_vd:,} -> ít hơn {S_vd * (S_vd + 1) / 2 / (S_vd * cua_so_vd):.1f} lần "
      f"(cửa sổ {cua_so_vd:,} là lựa chọn của sổ tay này)")
print("mặt nạ cửa sổ 4 trên 8 vị trí (1 là được nhìn):")
print(mat_na(0, 8, 8, 4).int().numpy())
for ten, cs in (("cửa sổ 4 ở cả hai tầng", (4, 4)), ("cửa sổ 4, rồi dày đặc", (4, 0))):
    dat_hat_giong()
    duong_nl[ten], giay = huan_luyen(MoHinh(V_NL, cua_so=cs), 1000)
    do_tren_may(f"{ten}: độ chính xác sau 1,000 bước", round(duong_nl[ten][-1], 3))
    do_tren_may(f"{ten}: thời gian", round(giay, 1), "s")

fig, ax = plt.subplots(figsize=(6.8, 3.6))
for ten, mau in (("nhiều đầu", MAU[0]), ("cửa sổ 4, rồi dày đặc", MAU[2]), ("cửa sổ 4 ở cả hai tầng", MAU[1])):
    ax.plot(buoc, duong_nl[ten], "o-", color=mau, label="dày đặc ở cả hai tầng" if ten == "nhiều đầu" else ten)
ax.axhline(1 / P, color="0.5", lw=0.8, ls="--", label="đoán bừa trong ngữ cảnh: 1/8")
ax.set_ylim(0, 1.05)
ax.set_xlabel("bước huấn luyện")
ax.set_ylabel("độ chính xác ở câu hỏi")
ax.set_title("Bài nhớ lại với cửa sổ trượt (đo trên máy này)")
ax.legend(loc="center right", bbox_to_anchor=(1, 0.62), fontsize=8)
plt.tight_layout()
plt.show()

Với cửa sổ ở cả hai tầng, một câu hỏi chỉ "thấy" được chừng sáu token trước nó sau hai tầng, nên cặp nằm xa hơn là vô
hình. Mô hình chỉ học được những câu hỏi mà cặp được hỏi nằm gần, nên đường của nó bò lên chậm và có trần: huấn luyện
thêm bao lâu cũng không với tới những cặp ở đầu ngữ cảnh. Đặt một tầng dày đặc sau tầng cửa sổ là đủ, và còn học
nhanh hơn mô hình dày đặc hoàn toàn: tầng cửa sổ gắn mỗi giá trị với khoá ngay trước nó, rồi tầng dày đặc tra cứu trên khắp
ngữ cảnh. Đó là lý do các mô hình xen kẽ hai loại tầng thay vì chỉ dùng cửa sổ.

## 6. Tập trung tuyến tính: một RNN trá hình

Tập trung tuyến tính (linear attention) thay $\exp(q \cdot k)$ bằng tích của hai hàm đặc trưng dương,
$\phi(q)^\top \phi(k)$, ở đây $\phi(x) = \text{elu}(x) + 1$. Khi đó phép nhân có thể đổi thứ tự:

$$o_t = \frac{\phi(q_t)^\top \sum_{s \le t} \phi(k_s)\, v_s^\top}{\phi(q_t)^\top \sum_{s \le t} \phi(k_s)} .$$

Tổng ở tử là một ma trận $d \times d$ cộng dồn qua từng token. Nghĩa là có thể tính nó như một RNN: giữ một trạng
thái cố định, cập nhật nó mỗi token. Không còn ma trận $S \times S$, không còn bộ nhớ đệm phình ra. Chương nêu tập
trung tuyến tính trong số các cách xấp xỉ; công thức và mọi phép đo ở đây là của sổ tay này.

**Dự đoán:** cách tính song song (cả chuỗi một lúc) và cách tính hồi quy (từng token một) có cho cùng kết quả không?

In [ ]:
def tuyen_tinh_song_song(q, k, v):
    """Nhân quả, dạng song song: cộng dồn φ(k)vᵀ theo vị trí."""
    fq, fk = F.elu(q) + 1, F.elu(k) + 1
    tu = torch.einsum("htd,hte->htde", fk, v).cumsum(1)        # (đầu, T, d, d)
    mau = fk.cumsum(1)
    return torch.einsum("htd,htde->hte", fq, tu) / torch.einsum("htd,htd->ht", fq, mau)[..., None]


def tuyen_tinh_hoi_quy(q, k, v):
    """Cùng phép tính, từng token một, với một trạng thái d×d cố định cho mỗi đầu."""
    H, T, d = q.shape
    trang_thai, chuan = torch.zeros(H, d, d), torch.zeros(H, d)
    ra = []
    for t in range(T):
        fq, fk = F.elu(q[:, t]) + 1, F.elu(k[:, t]) + 1
        trang_thai = trang_thai + fk[:, :, None] * v[:, t, None, :]
        chuan = chuan + fk
        ra.append(torch.einsum("hd,hde->he", fq, trang_thai) / (fq * chuan).sum(-1, keepdim=True))
    return torch.stack(ra, 1)


g = torch.Generator().manual_seed(SEED)
q, k, v = (torch.randn(4, 128, 16, generator=g, dtype=torch.float64) for _ in range(3))
print(f"song song so với hồi quy: lệch lớn nhất {(tuyen_tinh_song_song(q, k, v) - tuyen_tinh_hoi_quy(q, k, v)).abs().max():.1e}")

RNN_CHIEU = sach(512, nguon="2 KB · trạng thái 512 chiều")
theo_sach("trạng thái RNN 512 chiều, FP32 (KB)", RNN_CHIEU * FP32 / 1024, sach=2, nguon="2 KB · trạng thái 512 chiều")
trang_thai_tt = N_L * N_DAU * (D_DAU * D_DAU + D_DAU) * FP16
print(f"tập trung tuyến tính, ví dụ 32 tầng, 32 đầu, đầu {D_DAU} chiều: trạng thái {trang_thai_tt / 1e6:.1f} MB, "
      f"bất kể ngữ cảnh dài bao nhiêu")
print(f"bộ nhớ đệm KV của cùng mô hình: {bo_dem_kv(1, N_DAU) / 1e6:.2f} MB mỗi token, nên vượt trạng thái đó từ "
      f"token thứ {math.ceil(trang_thai_tt / bo_dem_kv(1, N_DAU)):,}")

Hai cách cho cùng kết quả tới sai số làm tròn. Trạng thái của tập trung tuyến tính giống trạng thái ẩn mà chương tính
cho RNN: cố định, bất kể chuỗi dài 10 hay 10,000 token. Thời gian cũng vậy: cách song song (không nhân quả, để so
với softmax cho công bằng) chỉ tăng tuyến tính theo $S$.

In [ ]:
def tuyen_tinh_khong_nhan_qua(q, k, v):
    fq, fk = F.elu(q) + 1, F.elu(k) + 1
    kv = torch.einsum("htd,hte->hde", fk, v)                   # tổng một lần, d×d mỗi đầu
    return torch.einsum("htd,hde->hte", fq, kv) / torch.einsum("htd,hd->ht", fq, fk.sum(1))[..., None]


thoi_gian_tt, thoi_gian_sm = {}, {}
g = torch.Generator().manual_seed(SEED)
for S in (512, 1024, 2048, 4096, 8192, 16384, 32768):
    q, k, v = (torch.randn(4, S, d_io, generator=g) for _ in range(3))
    lap = 3 if S <= 2048 else 1
    thoi_gian_tt[S] = thoi_gian(lambda: tuyen_tinh_khong_nhan_qua(q, k, v), lap=lap)
    if S <= 8192:
        thoi_gian_sm[S] = thoi_gian(lambda: F.scaled_dot_product_attention(q[None], k[None], v[None]), lap=lap)
    print(f"S = {S:>6,}: softmax {1e3 * thoi_gian_sm[S] if S in thoi_gian_sm else float('nan'):>8.1f} ms, "
          f"tuyến tính {1e3 * thoi_gian_tt[S]:>6.1f} ms")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.plot(list(thoi_gian_sm), [1e3 * t for t in thoi_gian_sm.values()], "o-", color=MAU[1],
        label="softmax (scaled_dot_product_attention)")
ax.plot(list(thoi_gian_tt), [1e3 * t for t in thoi_gian_tt.values()], "o-", color=MAU[0], label="tuyến tính")
truc_log(ax)
ax.set_xlabel("độ dài chuỗi S (token)")
ax.set_ylabel("thời gian (ms)")
ax.set_title("4 đầu, d = 64, không nhân quả (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Trên trục log, đường softmax dốc gấp đôi đường tuyến tính: bậc hai so với bậc một. Nghe như được cả đôi đường, nên
câu hỏi là mất gì. Một trạng thái $d \times d$ cố định phải nén mọi cặp khoá-giá trị đã gặp vào một số ô nhớ không
đổi, trong khi tập trung softmax giữ nguyên từng khoá. Ô dưới chạy lại bài nhớ lại với tập trung tuyến tính.

**Dự đoán:** nó học được bài nhớ lại không?

In [ ]:
dat_hat_giong()
duong_nl["tuyến tính"], giay = huan_luyen(MoHinh(V_NL, kieu="tuyen_tinh"), 1000)
do_tren_may("tập trung tuyến tính: độ chính xác sau 1,000 bước", round(duong_nl["tuyến tính"][-1], 3))
do_tren_may("tập trung tuyến tính: thời gian", round(giay, 1), "s")

fig, ax = plt.subplots(figsize=(6.8, 3.6))
ax.plot(buoc, duong_nl["nhiều đầu"], "o-", color=MAU[0], label="softmax, dày đặc")
ax.plot(buoc, duong_nl["tuyến tính"], "o-", color=MAU[4], label="tuyến tính, φ = elu + 1")
ax.axhline(1 / P, color="0.5", lw=0.8, ls="--", label="đoán bừa trong ngữ cảnh: 1/8")
ax.set_ylim(0, 1.05)
ax.set_xlabel("bước huấn luyện")
ax.set_ylabel("độ chính xác ở câu hỏi")
ax.set_title("Bài nhớ lại: softmax so với tuyến tính (đo trên máy này)")
ax.legend(loc="center right", fontsize=8)
plt.tight_layout()
plt.show()

Mô hình tuyến tính dừng ở mức thấp ngay từ đầu và không nhích lên: nó biết câu trả lời là một giá trị trong ngữ
cảnh, nhưng không tách được giá trị đúng ra khỏi trạng thái nén. Đó cũng là điểm yếu của RNN mà chương mô tả, chỉ khác ở chỗ huấn luyện được song song.
Các biến thể mới hơn của họ này (thêm cổng quên, trạng thái lớn hơn) đều nhắm vào đúng chỗ đó.

## Thử thêm

1. Ở phần 1, gọi `tap_trung_chia_khoi(q, k, v, 512, BoNhoCham())` với $S = 1{,}024$. Tỉ số lưu lượng là bao nhiêu,
   và bộ nhớ nhanh cần bao nhiêu KB? Vì sao không ai chọn khối to bằng cả chuỗi?
2. Ở phần 5, thử `cua_so=(0, 4)`: tầng dày đặc trước, tầng cửa sổ sau. Kết quả khác `(4, 0)` thế nào, và vì sao thứ
   tự lại quan trọng?
3. Ở phần 6, giảm `P` xuống 2 (chỉ 2 cặp trong ngữ cảnh), chạy lại ô `recall` rồi ô `linear_recall`. Tập trung
   tuyến tính có theo kịp không?

## Tóm lại

* Chia khối kiểu FlashAttention cho đúng kết quả của cách ngây thơ mà không bao giờ cần ma trận $S \times S$; ở 32k
  token, riêng ma trận đó đã là hàng gigabyte mỗi tầng, và bộ nhớ điểm tăng 16 lần khi chuỗi dài từ 512 lên 2,048.
* Bộ nhớ đệm KV biến mỗi bước sinh token thành việc chỉ xử lý token mới, đổi lại
  $N_L \times 2 \times N_{\text{heads}} \times S \times d_{\text{head}}$ phần tử cho mỗi yêu cầu: chừng 1 GB ở ví dụ
  32 tầng, 32 đầu, 2,048 token của chương, và hơn 100 GB ở ngữ cảnh dài.
* MQA và GQA chia số đầu khoá và giá trị, nên chia bộ nhớ đệm theo đúng tỉ lệ đó mà không giảm phép tính của điểm; một
  mô hình nhiều đầu đã học có thể gộp đầu rồi huấn luyện tiếp thay vì học lại từ đầu.
* Tập trung cửa sổ trượt cắt cả điểm lẫn bộ nhớ đệm, nhưng cần tầng dày đặc xen giữa để thông tin đi xa, như GPT-3
  xen kẽ hai loại tầng.
* Tập trung tuyến tính có trạng thái cố định như RNN và thời gian tăng tuyến tính, nhưng nhớ kém hơn hẳn tập trung
  softmax ở bài tra cứu.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kiến trúc mạng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch06-nn-architectures/), dựng từ chương
[*Network Architectures*](https://mlsysbook.ai/vol1/nn_architectures/nn_architectures.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.